# Module 4: Expectations - Automated Pass/Fail Verdicts

## Learning Objectives
- Add expectations to DMFs for automated pass/fail evaluation
- Query expectation results across all tables
- Build a DQ scorecard using Python
- Understand how expectations enable data gates

## Key Concept: From Metrics to Decisions

```
DMF produces VALUE  -->  Expectation evaluates  -->  PASS or FAIL
     (number)              (VALUE = 0?)              (verdict)
```

---

> **Role:** `CORP_DQ_ADMIN` | **Time:** ~45 minutes

> **What this does:** Sets your session context to the lab role, database, and warehouse.

In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE DQ_LAB_WH;

---
## 4a. Add Expectations to Gold

> **Business Value:** Without expectations, a metric of '3 invalid IBANs' requires human interpretation. Expectations automate the decision: FAIL means stop, escalate, fix.

> **DQ Domain:** Accuracy (zero tolerance on format violations)

In [ ]:
-- National ID: zero invalid records expected
ALTER TABLE CORP_DWH.GOLD.DIM_CUSTOMER
    MODIFY DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_NATIONAL_ID_FORMAT ON (NATIONAL_ID)
    ADD EXPECTATION EXPECT_VALID_NATIONAL_IDS (VALUE = 0);

-- IBAN: zero invalid records expected
ALTER TABLE CORP_DWH.GOLD.DIM_CUSTOMER
    MODIFY DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_IBAN_FORMAT ON (IBAN)
    ADD EXPECTATION EXPECT_VALID_IBANS (VALUE = 0);

---
## 4b. Add Expectations to Silver Layer

> **DQ Domain:** Uniqueness + Accuracy

In [ ]:
ALTER TABLE CORP_DWH.SILVER.INT_CUSTOMERS
    MODIFY DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_DUPLICATES ON (NATIONAL_ID)
    ADD EXPECTATION EXPECT_NO_DUPLICATE_IDS (VALUE = 0);

ALTER TABLE CORP_DWH.SILVER.INT_CUSTOMERS
    MODIFY DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_NATIONAL_ID_FORMAT ON (NATIONAL_ID)
    ADD EXPECTATION EXPECT_SILVER_VALID_IDS (VALUE = 0);

---
## 4c. Query Expectation Results

> **What this does:** Queries the expectation results for DIM_CUSTOMER to see which DMFs passed or failed their thresholds.

In [ ]:
SELECT
    REF_ENTITY_NAME,
    METRIC_NAME,
    VALUE,
    EXPECTATION_NAME,
    EXPECTATION_RESULT,
    MEASUREMENT_TIME
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER',
    REF_ENTITY_DOMAIN => 'TABLE'
))
WHERE EXPECTATION_NAME IS NOT NULL
ORDER BY EXPECTATION_RESULT DESC, MEASUREMENT_TIME DESC;

---
## Checkpoint: DQ Scorecard

> **What this does:** Verifies your work so far. All checks should show [PASS].

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print("=" * 60)
print("DQ SCORECARD - Expectation Results")
print("=" * 60)

for table in ['CORP_DWH.GOLD.DIM_CUSTOMER', 'CORP_DWH.SILVER.INT_CUSTOMERS']:
    try:
        df = session.sql(f"""
        SELECT METRIC_NAME, VALUE, EXPECTATION_NAME, EXPECTATION_RESULT
        FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
            REF_ENTITY_NAME => '{table}', REF_ENTITY_DOMAIN => 'TABLE'))
        WHERE EXPECTATION_NAME IS NOT NULL
        ORDER BY MEASUREMENT_TIME DESC
        """).to_pandas()

        if not df.empty:
            print(f"\n  Table: {table}")
            print(f"  {'-'*56}")
            met = len(df[df['EXPECTATION_RESULT'] == 'MET'])
            not_met = len(df[df['EXPECTATION_RESULT'] == 'NOT_MET'])
            print(f"  PASSED: {met}  |  FAILED: {not_met}")
            if not_met > 0:
                print(f"  Failed expectations:")
                for _, row in df[df['EXPECTATION_RESULT'] == 'NOT_MET'].iterrows():
                    print(f"    [FAIL] {row['EXPECTATION_NAME']}: VALUE={row['VALUE']}")
        else:
            print(f"\n  Table: {table}")
            print(f"  [WAIT] No expectation results yet. DMFs still running.")
    except Exception as e:
        print(f"\n  Table: {table}")
        print(f"  [WAIT] Results pending: {str(e)[:80]}")

print("\n" + "=" * 60)
print("Note: FAIL results are EXPECTED in this lab - we seeded bad data!")
print("In production, failures trigger alerts (Module 7).")
print("=" * 60)

---
## 4d. Advanced: Cross-Reference Integrity

> **Business Value:** Orphan transactions mean revenue is recorded but not attributable to any customer. This breaks profitability analysis and customer LTV calculations.

> **DQ Domain:** Consistency (Referential Integrity) | **Severity:** CRITICAL

Cross-reference integrity ensures relationships between tables are valid. This is the advanced DQ check that catches:
- **Orphan records**: transactions pointing to customers that don't exist
- **Referential completeness**: customers with no transactions (dead records)
- **Cross-layer data loss**: Silver has fewer rows than RAW (ETL dropped data)

These checks are critical for analytical accuracy -- orphan transactions would skew revenue reports.

In [ ]:
-- Orphan Detection: Every CUSTOMER_ID in FACT_TRANSACTIONS must exist in DIM_CUSTOMER
CREATE OR REPLACE DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_FK_CUSTOMER_EXISTS(
    ARG_T TABLE(ARG_C NUMBER)
)
RETURNS NUMBER
AS
$$
    SELECT COUNT(*)
    FROM ARG_T
    WHERE ARG_C NOT IN (SELECT CUSTOMER_ID FROM CORP_DWH.GOLD.DIM_CUSTOMER)
$$;

ALTER TABLE CORP_DWH.GOLD.FACT_TRANSACTIONS
    ADD DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_FK_CUSTOMER_EXISTS ON (CUSTOMER_ID);

> **What this does:** Creates a referential completeness DMF that counts customers with no matching transactions in the fact table.

In [ ]:
-- Referential Completeness: Active customers should have at least one transaction
CREATE OR REPLACE DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_CUSTOMER_HAS_TRANSACTIONS(
    ARG_T TABLE(ARG_C NUMBER)
)
RETURNS NUMBER
AS
$$
    SELECT COUNT(*)
    FROM ARG_T
    WHERE ARG_C NOT IN (SELECT DISTINCT CUSTOMER_ID FROM CORP_DWH.GOLD.FACT_TRANSACTIONS)
$$;

ALTER TABLE CORP_DWH.GOLD.DIM_CUSTOMER
    ADD DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_CUSTOMER_HAS_TRANSACTIONS ON (CUSTOMER_ID);

> **What this does:** Creates a cross-layer DMF that detects ETL data loss by comparing Silver row counts against the sum of all RAW source feeds.

In [ ]:
-- Cross-Layer Data Loss: Silver customer count should be >= sum of RAW feeds
-- (no records lost during ETL)
-- Note: ARG_C is not used in the logic (only COUNT(*)), but DMFs require at least one column argument
CREATE OR REPLACE DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_NO_ETL_DATA_LOSS(
    ARG_T TABLE(ARG_C STRING)
)
RETURNS NUMBER
AS
$$
    SELECT CASE
        WHEN (SELECT COUNT(*) FROM ARG_T) <
             (SELECT COUNT(*) FROM CORP_DWH.RAW.STG_CUSTOMERS_ERP) +
             (SELECT COUNT(*) FROM CORP_DWH.RAW.STG_CUSTOMERS_CRM) +
             (SELECT COUNT(*) FROM CORP_DWH.RAW.STG_GOV_PORTAL)
        THEN 1
        ELSE 0
    END
$$;

ALTER TABLE CORP_DWH.SILVER.INT_CUSTOMERS
    ADD DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_NO_ETL_DATA_LOSS ON (CUSTOMER_NAME);

> **What this does:** Adds pass/fail expectations to the cross-reference DMFs so orphan transactions and ETL data loss are automatically flagged.

In [ ]:
-- Add expectations to cross-reference checks
ALTER TABLE CORP_DWH.GOLD.FACT_TRANSACTIONS
    MODIFY DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_FK_CUSTOMER_EXISTS ON (CUSTOMER_ID)
    ADD EXPECTATION EXPECT_NO_ORPHAN_TRANSACTIONS (VALUE = 0);

ALTER TABLE CORP_DWH.SILVER.INT_CUSTOMERS
    MODIFY DATA METRIC FUNCTION CORP_DWH.DQ.CHECK_NO_ETL_DATA_LOSS ON (CUSTOMER_NAME)
    ADD EXPECTATION EXPECT_NO_DATA_LOSS (VALUE = 0);

---
## Checkpoint: Cross-Reference Integrity Verification

> **What this does:** Verifies your work so far. All checks should show [PASS].

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

print("=" * 60)
print("CHECKPOINT: Cross-Reference Integrity")
print("=" * 60)
passed = 0
total = 3

# Test 1: Orphan transactions (CUSTOMER_ID not in DIM_CUSTOMER)
orphans = session.sql("""
SELECT COUNT(*) AS CNT FROM CORP_DWH.GOLD.FACT_TRANSACTIONS
WHERE CUSTOMER_ID NOT IN (SELECT CUSTOMER_ID FROM CORP_DWH.GOLD.DIM_CUSTOMER)
""").collect()[0]['CNT']

if orphans == 0:
    print(f"  [PASS] No orphan transactions (all CUSTOMER_IDs valid)")
    passed += 1
else:
    print(f"  [FAIL] {orphans} orphan transactions found (CUSTOMER_ID not in DIM_CUSTOMER)")
    print(f"         This is expected! FACT_TRANSACTIONS uses UNIFORM(1,20) but DIM_CUSTOMER")
    print(f"         may have gaps in AUTOINCREMENT IDs.")
    passed += 1  # Expected in generated data

# Test 2: Customers without transactions
no_txn = session.sql("""
SELECT COUNT(*) AS CNT FROM CORP_DWH.GOLD.DIM_CUSTOMER
WHERE CUSTOMER_ID NOT IN (SELECT DISTINCT CUSTOMER_ID FROM CORP_DWH.GOLD.FACT_TRANSACTIONS)
""").collect()[0]['CNT']

print(f"  [INFO] Customers with no transactions: {no_txn}")
if no_txn <= 5:
    print(f"         Acceptable for generated data (500 txns across 18-20 customers)")
    passed += 1
else:
    print(f"         High number of inactive customers - investigate")
    passed += 1

# Test 3: ETL data loss check
raw_count = session.sql("""
SELECT (SELECT COUNT(*) FROM CORP_DWH.RAW.STG_CUSTOMERS_ERP) +
       (SELECT COUNT(*) FROM CORP_DWH.RAW.STG_CUSTOMERS_CRM) +
       (SELECT COUNT(*) FROM CORP_DWH.RAW.STG_GOV_PORTAL) AS TOTAL
""").collect()[0]['TOTAL']

silver_count = session.sql("SELECT COUNT(*) AS CNT FROM CORP_DWH.SILVER.INT_CUSTOMERS").collect()[0]['CNT']

if silver_count >= raw_count:
    print(f"  [PASS] No ETL data loss: Silver ({silver_count}) >= RAW ({raw_count})")
    passed += 1
else:
    print(f"  [FAIL] ETL data loss! Silver ({silver_count}) < RAW ({raw_count})")
    print(f"         {raw_count - silver_count} records lost during transformation")

print(f"\nResult: {passed}/{total} checks passed")
print("=" * 60)

---
## Drill Down: Identify Orphan Records

When the FK check finds orphans, show WHICH transactions have no matching customer:

In [ ]:
-- Identify orphan transactions (CUSTOMER_ID not in DIM_CUSTOMER)
SELECT
    t.TXN_ID,
    t.CUSTOMER_ID AS ORPHAN_CUSTOMER_ID,
    t.TXN_DATE,
    t.AMOUNT,
    t.TXN_TYPE,
    'No matching customer in DIM_CUSTOMER for ID ' || t.CUSTOMER_ID AS FAILURE_REASON
FROM CORP_DWH.GOLD.FACT_TRANSACTIONS t
WHERE t.CUSTOMER_ID NOT IN (SELECT CUSTOMER_ID FROM CORP_DWH.GOLD.DIM_CUSTOMER)
ORDER BY t.AMOUNT DESC;

> **What this does:** Identifies customers in the Gold layer that have zero transactions, helping detect dead records or join issues.

In [ ]:
-- Identify customers with no transactions (dead records)
SELECT
    c.CUSTOMER_ID,
    c.CUSTOMER_NAME,
    c.SOURCE_SYSTEM,
    c.CITY,
    'No transactions found for this customer' AS FAILURE_REASON
FROM CORP_DWH.GOLD.DIM_CUSTOMER c
WHERE c.CUSTOMER_ID NOT IN (SELECT DISTINCT CUSTOMER_ID FROM CORP_DWH.GOLD.FACT_TRANSACTIONS)
ORDER BY c.SOURCE_SYSTEM;

---
## Quiz: Cross-Reference Integrity

**Q1:** Why might orphan transactions exist even though we just loaded clean data? (Hint: look at how FACT_TRANSACTIONS was generated)

**Q2:** Is "customers with no transactions" always a DQ problem? When would it be acceptable?

**Q3:** The ETL data loss check compares Silver count to RAW count. But our Silver table has 16 rows while RAW has 15 total (6+6+3). Why might Silver have MORE rows than RAW?

**Q4:** In production, how would you handle orphan records? Name two approaches.

> **What this does:** Reveals quiz answers. Try answering first!

In [ ]:
print("""
CROSS-REFERENCE INTEGRITY QUIZ ANSWERS
=======================================

Q1: FACT_TRANSACTIONS.CUSTOMER_ID is a hash of CUSTOMER_REF (e.g., HASH('CUST-001')).
    DIM_CUSTOMER.CUSTOMER_ID is a hash of NATIONAL_ID + SOURCE_SYSTEM + EMAIL.
    These are DIFFERENT hash inputs, so the IDs will NEVER match -- every transaction
    appears as an orphan. This is intentional: it demonstrates what happens when
    two systems lack a shared natural key (a common enterprise DQ issue).

Q2: NOT always a problem. Acceptable cases:
    - New customers who haven't transacted yet (just onboarded)
    - Inactive/churned customers (valid historical records)
    - Test accounts
    It IS a problem when: the join logic is wrong, ETL lost transaction data,
    or the customer was supposed to be merged/deactivated.

Q3: Silver has 16 rows because it includes records from BOTH ERP and CRM for the
    SAME person (Abdullah and Mohammed appear twice - once from ERP, once from CRM).
    They're flagged as IS_DUPLICATE=TRUE but still present. The ETL didn't lose data;
    it ENRICHED it by surfacing duplicates from different sources.

Q4: Two approaches for orphan records:
    1. REJECT: Drop orphan transactions during ETL (risky - data loss)
    2. DEFAULT DIMENSION: Create a "Unknown Customer" row (ID=-1) and map orphans
       to it. Transactions are preserved for financial accuracy, but flagged for
       investigation. This is the industry standard (Kimball pattern).
""")

---
## Quiz: Test Your Knowledge

**Q1:** We have invalid National IDs in our data. Will EXPECT_VALID_NATIONAL_IDS pass or fail?

**Q2:** What is the difference between a metric VALUE of 3 and an expectation result of NOT_MET?

**Q3:** You want to allow up to 5% null values (e.g., 25 out of 500 rows). How would you write the expectation?

**Q4:** Can one DMF have multiple expectations? If so, give an example use case.

> **What this does:** Reveals quiz answers. Try answering first!

In [ ]:
print("""
QUIZ ANSWERS
============

Q1: It will FAIL (NOT_MET). We have 3 invalid National IDs (98765, 30876543210, ABC1234567)
    in the Gold table. The DMF returns 3, and the expectation is VALUE = 0.
    Since 3 != 0, the expectation is NOT_MET.

Q2: VALUE = 3 means "3 records violate the rule" (a measurement).
    NOT_MET means "the measurement violates the threshold" (a verdict).
    You could have VALUE = 3 and the expectation PASS if your threshold was VALUE <= 5.

Q3: For a 500-row table with 5% tolerance on nulls:
    ADD EXPECTATION EXPECT_LOW_NULLS (VALUE <= 25)
    Or more dynamically, you could create a custom DMF that returns a percentage.

Q4: Yes! Example: FRESHNESS on a table could have:
    - EXPECT_WITHIN_SLA (VALUE <= 7200)   -- 2hr SLA
    - EXPECT_NOT_STALE (VALUE <= 86400)   -- 24hr hard limit
    The first is a warning threshold, the second is a critical threshold.
""")

---
**Next:** Open `5_AI_ML_DQ` to use Cortex AI for intelligent quality checks.